In [21]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq 
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy , ToolStrategy
from pydantic import BaseModel , Field

load_dotenv()

True

In [3]:
model = ChatGroq(model = 'openai/gpt-oss-120b')

In [8]:
prompt = """

Understand the coding request :

For any request , identify the following 

-Task type
-objective 
-file involved 

Return the response in the following json format :

{

    'task_type' : "type of task",
    "objective" : "objective of the task",
    "files_involved" : "list of all the files involved"
}

"""

In [5]:
request = "Identify the bug in the auth.py file where the exchange token is not gettig generated "

In [9]:
response = model.invoke(prompt+ request)
print(response.content)

{
    "task_type": "debugging",
    "objective": "Identify and explain the bug in the auth.py file that prevents the exchange token from being generated.",
    "files_involved": ["auth.py"]
}


In [12]:
from typing import Literal


class CodingRequest(BaseModel):
    """Structured rep of the coding request """

    task_type:Literal[
        "implement",
        "debug",
        "review",
        "explain"
    ] = Field(description = "Type of the task performed ")

    objective : str = Field(description= "A consice description of what needs to be achieved")

    need_code_changes:bool = Field(description= "whether fullfilling this task requires modifing the ource code")
    target_files : list[str] = Field("List of files that are relevnt to the task eg: ['auth.py' , 'models.py]")

In [ ]:
structured_output_model = model.with_structured_output(CodingRequest)
# read the doc of with_structured_output

In [16]:
result = structured_output_model.invoke(""""
fix the login bug in auty.py . Expired session currently produce http 500.
                            """)

In [ ]:
print(type(result))
print(result)


<class '__main__.CodingRequest'>


In [19]:
CodingRequest.model_json_schema()

{'description': 'Structured rep of the coding request ',
 'properties': {'task_type': {'description': 'Type of the task performed ',
   'enum': ['implement', 'debug', 'review', 'explain'],
   'title': 'Task Type',
   'type': 'string'},
  'objective': {'description': 'A consice description of what needs to be achieved',
   'title': 'Objective',
   'type': 'string'},
  'need_code_changes': {'description': 'whether fullfilling this task requires modifing the ource code',
   'title': 'Need Code Changes',
   'type': 'boolean'},
  'target_files': {'default': "List of files that are relevnt to the task eg: ['auth.py' , 'models.py]",
   'items': {'type': 'string'},
   'title': 'Target Files',
   'type': 'array'}},
 'required': ['task_type', 'objective', 'need_code_changes'],
 'title': 'CodingRequest',
 'type': 'object'}

In [22]:
agent  = create_agent(
    model = model,
    tools = [],
    response_format= CodingRequest,
    system_prompt=prompt
    
    
)

In [25]:
result = agent.invoke({
    "messages" : [
        {
            "role" : "user",
            "content" : "Fix the login bug in the auth.py expired session currently produce HTTP 500"
        }
        ]
})

print(result["structured_response"])

task_type='debug' objective='Resolve the HTTP 500 error caused by expired sessions during login in auth.py' need_code_changes=True target_files=['auth.py']
